# U.S. Treasury Curve Construction, Bond Risk, and Stress Testing

**Scope.** This notebook downloads Treasury constant-maturity yields from FRED, fits three discount curves (Nelson–Siegel, Nelson–Siegel–Svensson, and a par-yield bootstrap), validates them out of sample, prices a synthetic fixed-rate portfolio, computes duration, convexity and key-rate DV01, and stress-tests the portfolio by full repricing. A separate check compares the curves with one published Treasury auction price. All outputs are saved to a timestamped folder and can be exported as a ZIP.

| Section | Content |
|---|---|
| 0 | Setup and configuration |
| 1 | FRED data, snapshot and coverage |
| 2 | Curve models, in-sample fit, forwards, held-out validation |
| 3 | Bond pricing, risk measures, Treasury auction price check |
| 4 | Stress engine and interactive playground |
| 5 | Publication figures and export bundle |

**Run order.** Use *Runtime → Run all* once, top to bottom. The playground in Section 4 needs the engine cell before it. Section 5 needs every earlier section.

**Conventions.**
- FRED quotes are percent, converted to decimals. Nothing is imputed or forward-filled.
- FRED constant-maturity yields are par-style quotes, not zero rates. NS/NSS fit quotes through a semiannual par-yield mapping. Quotes under one year are treated as simple-interest yields.
- Zero rates are continuously compounded. Time is in years.
- The stress portfolio is synthetic and its scenarios are illustrative, not regulatory specifications.

## 0. Setup and configuration

Every user-adjustable setting is in the next cell.

In [ ]:
import hashlib
import importlib.metadata as importlib_metadata
import json
import shutil
import sys
import zipfile
from dataclasses import dataclass
from datetime import datetime, timezone
from io import BytesIO
from itertools import product
from pathlib import Path
from urllib.parse import urlsplit  # noqa: F401

import matplotlib as mpl
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import requests
from IPython.display import clear_output, display
from matplotlib.lines import Line2D
from matplotlib.patches import Patch
from matplotlib.ticker import StrMethodFormatter
from requests.adapters import HTTPAdapter
from scipy.optimize import brentq, least_squares
from urllib3.util.retry import Retry

# ---- Configuration -------------------------------------------------------
START_DATE = "2015-01-01"        # FRED download window (inclusive)
END_DATE = "2025-12-31"
FIT_DATE = "2019-12-31"          # Valuation date for Sections 2-4
PLOT_START = "2025-12-01"        # Window for the Section 1 curve plot
PLOT_END = "2025-12-31"
N_VALIDATION_DATES = 24          # Held-out validation dates
HOLDOUT_IDS = ["DGS2", "DGS7", "DGS20"]
AUCTION_START = "2025-01-01"     # Treasury auction search window
AUCTION_END = "2025-12-31"
REPORT_MODEL = "NS"              # Curve model used in the shape figure
DOWNLOAD_ZIP = True              # Trigger a browser download in Colab

FRED_CSV = "https://fred.stlouisfed.org/graph/fredgraph.csv"
AUCTIONS_API = (
    "https://api.fiscaldata.treasury.gov/services/api/fiscal_service/"
    "v1/accounting/od/auctions_query"
)

# FRED series ID -> maturity in years
TENORS = {
    "DGS1MO": 1 / 12, "DGS3MO": 3 / 12, "DGS6MO": 6 / 12, "DGS1": 1.0,
    "DGS2": 2.0, "DGS3": 3.0, "DGS5": 5.0, "DGS7": 7.0,
    "DGS10": 10.0, "DGS20": 20.0, "DGS30": 30.0,
}
CORE_IDS = list(TENORS)
MODELS = ("NS", "NSS", "Bootstrap")
COLORS = {"NS": "#0072B2", "NSS": "#D55E00", "Bootstrap": "#009E73"}

start_ts = pd.to_datetime(START_DATE, format="%Y-%m-%d", errors="raise")
end_ts = pd.to_datetime(END_DATE, format="%Y-%m-%d", errors="raise")
if start_ts > end_ts:
    raise ValueError("Supply valid dates with START_DATE <= END_DATE.")

run_time = datetime.now(timezone.utc)
run_id = run_time.strftime("%Y%m%dT%H%M%S%fZ")
run_dir = Path("data/fred") / f"{start_ts.date()}_to_{end_ts.date()}_{run_id}"
raw_dir = run_dir / "raw"
raw_dir.mkdir(parents=True, exist_ok=False)

print(f"Requested range: {start_ts.date()} to {end_ts.date()}")
print(f"Snapshot folder: {run_dir}")

## 1. Data: FRED Treasury constant-maturity yields

Each series is downloaded separately. The raw bytes, request URL, retrieval time and SHA-256 hash are stored so the snapshot can be audited later. The working panel keeps every date returned by any series. `core_panel` keeps only dates on which all eleven tenors are quoted, and the models use only those dates.

In [ ]:
def fetch_fred_series(session, series_id, start, end, raw_dir):
    """Download one FRED series and return (percent Series, provenance dict)."""
    params = {
        "id": series_id,
        "cosd": start.date().isoformat(),
        "coed": end.date().isoformat(),
    }
    response = session.get(FRED_CSV, params=params, timeout=(10, 45))
    response.raise_for_status()

    raw_bytes = response.content
    frame = pd.read_csv(BytesIO(raw_bytes), dtype="string", keep_default_na=False)
    frame.columns = frame.columns.str.strip()

    date_col = next(
        (c for c in frame.columns if c.lower() in {"date", "observation_date"}),
        None,
    )
    if date_col is None or series_id not in frame.columns:
        raise ValueError(f"{series_id}: unexpected CSV columns {frame.columns.tolist()}")

    dates = pd.to_datetime(frame[date_col].str.strip(), format="%Y-%m-%d", errors="raise")
    if dates.isna().any() or dates.duplicated().any():
        raise ValueError(f"{series_id}: missing or duplicate observation dates.")

    text = frame[series_id].str.strip().replace({"": pd.NA, ".": pd.NA})
    values = pd.to_numeric(text, errors="raise").to_numpy(dtype=float, na_value=np.nan)
    if np.isinf(values).any():
        raise ValueError(f"{series_id}: infinite rate found.")

    series = pd.Series(
        values, index=pd.DatetimeIndex(dates, name="date"), name=series_id
    ).sort_index().loc[start:end]

    raw_file = raw_dir / f"{series_id}.csv"
    raw_file.write_bytes(raw_bytes)
    metadata = {
        "series_id": series_id,
        "maturity_years": TENORS[series_id],
        "requested_url": response.request.url,
        "response_url": response.url,
        "retrieved_at_utc": datetime.now(timezone.utc).isoformat(),
        "raw_file": str(raw_file.relative_to(raw_dir.parent)),
        "sha256": hashlib.sha256(raw_bytes).hexdigest(),
    }
    return series, metadata


retry = Retry(
    total=3, backoff_factor=1,
    status_forcelist=(429, 500, 502, 503, 504), allowed_methods={"GET"},
)

series_list, downloads = [], []
with requests.Session() as session:
    session.mount("https://", HTTPAdapter(max_retries=retry))
    for series_id in TENORS:
        try:
            series, metadata = fetch_fred_series(session, series_id, start_ts, end_ts, raw_dir)
        except (requests.RequestException, ValueError) as exc:
            raise RuntimeError(f"Fetch or validation failed for {series_id}: {exc}") from exc
        series_list.append(series)
        downloads.append(metadata)
        print(f"{series_id:8s}  {series.notna().sum():>6,} non-missing observations")

yield_panel_pct = pd.concat(series_list, axis=1).sort_index()
yield_panel_pct.index.name = "date"
if yield_panel_pct.empty or not yield_panel_pct.notna().any().any():
    raise ValueError("No usable observations in this range. Check the requested dates.")

yield_panel = yield_panel_pct / 100.0               # decimal annual yields
core_panel = yield_panel[CORE_IDS].dropna(how="any")  # complete curves only
full_panel = yield_panel.dropna(how="any")
curves_by_date = {
    d.strftime("%Y-%m-%d"): {s: None if pd.isna(r) else float(r) for s, r in row.items()}
    for d, row in yield_panel.dropna(how="all").iterrows()
}

In [ ]:
active_panel = yield_panel.dropna(how="all")
all_empty_dates = len(yield_panel) - len(active_panel)

coverage = pd.DataFrame(
    {
        "maturity_years": pd.Series(TENORS),
        "n_observed": active_panel.notna().sum(),
        "n_missing": active_panel.isna().sum(),
        "missing_pct": 100 * active_panel.isna().mean(),
        "first_observed": [active_panel[s].first_valid_index() for s in TENORS],
        "last_observed": [active_panel[s].last_valid_index() for s in TENORS],
    },
    index=list(TENORS),
)
coverage.index.name = "series_id"

manifest = {
    "source": "FRED / Federal Reserve Board H.15",
    "requested_start": START_DATE,
    "requested_end": END_DATE,
    "source_units": "percent",
    "working_units": "decimal annual yield",
    "transformation": "percent / 100; no imputation",
    "core_series_ids": CORE_IDS,
    "counts": {
        "panel_rows": len(yield_panel),
        "all_empty_rows": all_empty_dates,
        "complete_core_dates": len(core_panel),
        "complete_full_dates": len(full_panel),
    },
    "downloads": downloads,
}

yield_panel_pct.to_csv(run_dir / "yields_percent.csv")
yield_panel.to_csv(run_dir / "yields_decimal.csv")
coverage.to_csv(run_dir / "tenor_coverage.csv")
(run_dir / "manifest.json").write_text(json.dumps(manifest, indent=2), encoding="utf-8")

print(f"Observed range: {active_panel.index.min().date()} to {active_panel.index.max().date()}")
print(f"Rows with no quotes: {all_empty_dates:,}")
print(f"Complete core-grid dates: {len(core_panel):,}")
print(f"Saved to: {run_dir.resolve()}")
display(coverage.round({"maturity_years": 4, "missing_pct": 2}))

In [ ]:
# Observed curves over the plot window. Dates without a full curve are excluded.
plot_panel = core_panel.loc[PLOT_START:PLOT_END]
if plot_panel.empty:
    raise ValueError(f"No complete Treasury curves between {PLOT_START} and {PLOT_END}.")

series_ids = list(TENORS)
maturities = [TENORS[s] for s in series_ids]
tenor_labels = ["1M", "3M", "6M", "1Y", "2Y", "3Y", "5Y", "7Y", "10Y", "20Y", "30Y"]

fig, ax = plt.subplots(figsize=(10, 5))
for d in plot_panel.index:
    ax.plot(maturities, plot_panel.loc[d, series_ids].to_numpy(float) * 100,
            marker="o", linewidth=1.7, label=d.strftime("%Y-%m-%d"))
ax.set_xscale("log")
ax.set_xticks(maturities)
ax.set_xticklabels(tenor_labels)
ax.minorticks_off()
ax.set(title="U.S. Treasury constant-maturity yield curves",
       xlabel="Maturity (log scale)", ylabel="Yield (%)")
ax.grid(alpha=0.25)
if len(plot_panel) <= 12:
    ax.legend(ncol=2, fontsize=8)
fig.tight_layout()
plt.show()

## 2. Curve models

Three curves are fitted to the same quotes on a single date.

- **Nelson–Siegel (NS)** and **Nelson–Siegel–Svensson (NSS)** are parametric zero curves. Parameters are chosen by bounded least squares on quote errors in basis points, using several starting points and keeping the best converged fit.
- **Bootstrap** solves node discount factors sequentially so the curve reprices every par quote exactly, with log-linear interpolation of discount factors. Its forward curve is piecewise constant.

Parameters touching a bound are reported, since that signals a weakly identified fit.

In [ ]:
class TreasuryCurveFit:
    """Fit an NS or NSS zero curve to one row of decimal FRED quotes."""

    def __init__(self, model, tenors):
        self.model = model.upper()
        if self.model not in {"NS", "NSS"}:
            raise ValueError("model must be 'NS' or 'NSS'")

        self.ids = sorted(tenors, key=tenors.get)
        self.t = np.array([tenors[s] for s in self.ids], dtype=float)
        self.names = ["beta0", "beta1", "beta2"]
        self.names += ["beta3", "tau1", "tau2"] if self.model == "NSS" else ["tau1"]

        # Semiannual coupon schedules for tenors of one year or longer.
        self.coupons = {
            i: np.arange(1, int(round(2 * T)) + 1) / 2
            for i, T in enumerate(self.t) if T >= 1
        }
        self.result_ = None

    @staticmethod
    def _loadings(t, tau):
        x = np.asarray(t, dtype=float) / tau
        level = np.divide(-np.expm1(-x), x, out=np.ones_like(x), where=x != 0)
        return level, level - np.exp(-x)

    def _zero(self, t, p):
        tau1 = p[-1] if self.model == "NS" else p[-2]
        a, b = self._loadings(t, tau1)
        z = p[0] + p[1] * a + p[2] * b
        if self.model == "NSS":
            _, c = self._loadings(t, p[-1])
            z = z + p[3] * c
        return z

    def _discount(self, t, p):
        t = np.asarray(t, dtype=float)
        return np.exp(-t * self._zero(t, p))

    def _quotes(self, p):
        """Model-implied quotes: simple yield below 1Y, semiannual par yield above."""
        T = self.t
        quotes = np.empty_like(T)
        short = T < 1
        quotes[short] = np.expm1(T[short] * self._zero(T[short], p)) / T[short]
        for i, times in self.coupons.items():
            dfs = self._discount(times, p)
            quotes[i] = -2 * np.expm1(-T[i] * self._zero(T[i], p)) / dfs.sum()
        return quotes

    def fit(self, row):
        self.result_ = None
        y = row.reindex(self.ids).to_numpy(dtype=float)
        if not np.isfinite(y).all():
            raise ValueError("Supply one complete curve row in decimal rates.")

        n_beta = 3 if self.model == "NS" else 4
        n_tau = 1 if self.model == "NS" else 2
        lo = np.array([-0.30] * n_beta + [0.05] * n_tau)
        hi = np.array([0.30] * n_beta + [30.0] * n_tau)
        beta = [y[-1], y[0] - y[-1], 0.0] + ([0.0] if n_beta == 4 else [])

        def residuals(p):
            return 10_000 * (self._quotes(p) - y)

        attempts = []
        for taus in product((0.5, 2.0, 7.0), repeat=n_tau):
            if n_tau == 2 and taus[0] == taus[1]:
                continue
            p0 = np.clip(np.array(beta + list(taus)), lo + 1e-8, hi - 1e-8)
            attempts.append(
                least_squares(residuals, p0, bounds=(lo, hi), x_scale="jac", max_nfev=1000)
            )

        converged = [r for r in attempts if r.success and np.isfinite(r.cost)]
        if not converged:
            raise RuntimeError(f"{self.model}: no starting point converged.")

        self.result_ = min(converged, key=lambda r: r.cost)
        p = self.result_.x
        self.params_ = pd.Series(p, index=self.names, name=self.model)
        self.n_converged_ = len(converged)
        self.n_attempts_ = len(attempts)
        near_bound = np.isclose(p, lo, atol=1e-5) | np.isclose(p, hi, atol=1e-5)
        self.near_bounds_ = self.params_.index[near_bound].tolist()

        fitted = self._quotes(p)
        self.diagnostics_ = pd.DataFrame(
            {"maturity_years": self.t, "observed": y, "fitted": fitted,
             "residual_bp": 10_000 * (fitted - y)},
            index=pd.Index(self.ids, name="series_id"),
        )
        return self

    def zero_rates(self, t):
        if self.result_ is None:
            raise RuntimeError("Call fit before requesting zero rates.")
        return self._zero(t, self.result_.x)

    def discount_factors(self, t):
        if self.result_ is None:
            raise RuntimeError("Call fit before requesting discount factors.")
        return self._discount(t, self.result_.x)

In [ ]:
def quote_from_discount(discount_factors, maturity):
    """Model-implied quote (decimal) under the same conventions as the fits."""
    T = float(maturity)
    if not np.isfinite(T) or T <= 0:
        raise ValueError("Maturity must be positive and finite.")
    if T < 1.0:
        return float((1.0 / discount_factors(T) - 1.0) / T)

    n = int(round(2 * T))
    if not np.isclose(n / 2, T, rtol=0, atol=1e-12):
        raise ValueError("Coupon maturity must lie on the semiannual grid.")
    times = np.arange(1, n + 1) / 2
    dfs = discount_factors(times)
    return float(2.0 * (1.0 - dfs[-1]) / dfs.sum())


class TreasuryBootstrapCurve:
    """Sequential par-quote bootstrap with log-linear discount factors."""

    def __init__(self, tenors):
        self.ids = sorted(tenors, key=tenors.get)
        self.t = np.array([tenors[s] for s in self.ids], dtype=float)
        if (not len(self.t) or not np.isfinite(self.t).all()
                or self.t[0] <= 0 or np.any(np.diff(self.t) <= 0)):
            raise ValueError("Tenors must be finite, positive, and unique.")
        self.nodes_ = None

    def fit(self, row):
        self.nodes_ = None
        y = row.reindex(self.ids).to_numpy(dtype=float)
        if not np.isfinite(y).all():
            raise ValueError("Supply a complete row of decimal quotes.")

        nodes = np.r_[0.0, self.t]
        log_df = np.zeros_like(nodes)

        for i, (T, rate) in enumerate(zip(self.t, y), start=1):
            if T < 1.0:
                if 1.0 + rate * T <= 0:
                    raise ValueError(f"Invalid short-end quote at {T:g}Y.")
                log_df[i] = -np.log1p(rate * T)
                continue

            n = int(round(2 * T))
            if not np.isclose(n / 2, T, rtol=0, atol=1e-12):
                raise ValueError(f"{T:g}Y is not on the semiannual grid.")
            times = np.arange(1, n + 1) / 2

            def par_price_error(candidate, i=i, rate=rate, times=times):
                trial = np.r_[log_df[:i], candidate]
                dfs = np.exp(np.interp(times, nodes[:i + 1], trial))
                return dfs[-1] + (rate / 2.0) * dfs.sum() - 1.0

            try:
                log_df[i] = brentq(par_price_error, -50.0, 50.0, xtol=1e-13)
            except ValueError as exc:
                raise RuntimeError(
                    f"No bootstrap root bracketed at {T:g}Y, quote={rate:.6%}."
                ) from exc

        self.nodes_ = nodes
        self.log_df_ = log_df
        self.segment_forwards_ = -np.diff(log_df) / np.diff(nodes)

        fitted = np.array([quote_from_discount(self.discount_factors, T) for T in self.t])
        self.diagnostics_ = pd.DataFrame(
            {"maturity_years": self.t, "observed": y, "fitted": fitted,
             "residual_bp": 10_000 * (fitted - y)},
            index=pd.Index(self.ids, name="series_id"),
        )
        return self

    def _check_times(self, t):
        if self.nodes_ is None:
            raise RuntimeError("Call fit before evaluating the curve.")
        t = np.asarray(t, dtype=float)
        if np.any(~np.isfinite(t) | (t < 0) | (t > self.nodes_[-1])):
            raise ValueError(f"Evaluate on [0, {self.nodes_[-1]:g}] years.")
        return t

    def discount_factors(self, t):
        t = self._check_times(t)
        return np.exp(np.interp(t, self.nodes_, self.log_df_))

    def zero_rates(self, t):
        t = self._check_times(t)
        log_df = np.interp(t, self.nodes_, self.log_df_)
        return np.divide(-log_df, t, out=np.full_like(t, self.segment_forwards_[0]), where=t > 0)

    def instantaneous_forwards(self, t):
        t = self._check_times(t)
        i = np.searchsorted(self.nodes_, t, side="right") - 1
        return self.segment_forwards_[np.clip(i, 0, len(self.segment_forwards_) - 1)]


def fit_curve(model, row, tenors):
    if model == "Bootstrap":
        return TreasuryBootstrapCurve(tenors).fit(row)
    return TreasuryCurveFit(model, tenors).fit(row)


def instantaneous_forward(curve, t):
    if isinstance(curve, TreasuryBootstrapCurve):
        return curve.instantaneous_forwards(t)
    t = np.asarray(t, dtype=float)
    p = curve.params_
    x = t / p["tau1"]
    f = p["beta0"] + (p["beta1"] + p["beta2"] * x) * np.exp(-x)
    if curve.model == "NSS":
        x2 = t / p["tau2"]
        f = f + p["beta3"] * x2 * np.exp(-x2)
    return f


def period_forward_cc(curve, start, end):
    """Continuously compounded forward rate between two future dates."""
    if not 0 <= start < end:
        raise ValueError("Require 0 <= start < end.")
    dfs = curve.discount_factors(np.array([start, end]))
    return float((np.log(dfs[0]) - np.log(dfs[1])) / (end - start))

### 2.1 One-date fit and residuals

The bootstrap must reprice every quote to numerical precision; the notebook stops if it does not.

In [ ]:
curve_dir = Path(run_dir) / "section_2_curves"
curve_dir.mkdir(parents=True, exist_ok=True)

fit_date = pd.Timestamp(FIT_DATE)
if fit_date not in core_panel.index:
    raise ValueError(f"No complete curve for {FIT_DATE}.")

row = core_panel.loc[fit_date]
curves = {name: fit_curve(name, row, TENORS) for name in MODELS}

REPRICE_TOL_BP = 1e-4
bootstrap_error = curves["Bootstrap"].diagnostics_["residual_bp"].abs().max()
if not np.isfinite(bootstrap_error) or bootstrap_error > REPRICE_TOL_BP:
    raise RuntimeError(f"Bootstrap repricing check failed: {bootstrap_error:.6g} bp.")
print(f"Bootstrap repricing check passed: max |error| = {bootstrap_error:.3e} bp")

fit_quotes = pd.concat(
    {name: c.diagnostics_ for name, c in curves.items()}, names=["model", "series_id"]
)
fit_summary = fit_quotes.groupby(level="model")["residual_bp"].agg(
    rmse_bp=lambda e: np.sqrt(np.mean(e**2)),
    max_abs_bp=lambda e: e.abs().max(),
).reindex(MODELS)
for name in ("NS", "NSS"):
    fit_summary.loc[name, "converged_starts"] = f"{curves[name].n_converged_}/{curves[name].n_attempts_}"
    fit_summary.loc[name, "near_bounds"] = ", ".join(curves[name].near_bounds_) or "None"

obs = curves["Bootstrap"].diagnostics_
tenor_labels = [s[3:].replace("MO", "M") if s.endswith("MO") else f"{s[3:]}Y" for s in obs.index]

fig, (ax, er) = plt.subplots(2, 1, figsize=(11, 7), gridspec_kw={"height_ratios": [2, 1]})
x = np.arange(len(obs))
for j, name in enumerate(MODELS):
    d = curves[name].diagnostics_
    ax.plot(d["maturity_years"], 100 * d["fitted"], marker=".", color=COLORS[name], label=name)
    er.bar(x + (j - 1) * 0.25, d["residual_bp"], width=0.25, color=COLORS[name], label=name)
ax.scatter(obs["maturity_years"], 100 * obs["observed"], facecolors="none",
           edgecolors="black", s=52, zorder=5, label="Observed FRED quotes")
ax.set_xscale("log")
ax.set_xticks(obs["maturity_years"], tenor_labels)
ax.minorticks_off()
ax.set(title=f"Observed and fitted quotes | {fit_date:%Y-%m-%d}",
       xlabel="Maturity (log scale)", ylabel="Yield (%)")
ax.legend()
er.axhline(0, color="black", linewidth=0.8)
er.set_xticks(x, tenor_labels)
er.set(xlabel="Quoted tenor", ylabel="Fitted - observed (bp)")
for a in (ax, er):
    a.grid(axis="y", alpha=0.25)
fig.tight_layout()
fig.savefig(curve_dir / "quote_fit_and_residuals.png", dpi=180, bbox_inches="tight")
plt.show()

fit_quotes.to_csv(curve_dir / "one_date_quotes.csv")
fit_summary.to_csv(curve_dir / "one_date_fit_summary.csv")
display(fit_summary.round(6))
display(pd.DataFrame({n: curves[n].params_ for n in ("NS", "NSS")}).round(6))

### 2.2 Zero curves and implied forwards

Implied forwards are model outputs, not forecasts. The bootstrap forward is a step function by construction.

In [ ]:
grid = np.linspace(0.0, max(TENORS.values()), 1501)
fig, (axz, axf) = plt.subplots(2, 1, figsize=(11, 7), sharex=True)
forward_summary = {}
curve_grid = pd.DataFrame({"maturity_years": grid})

for name, curve in curves.items():
    dfs = curve.discount_factors(grid)
    z = curve.zero_rates(grid)
    f = instantaneous_forward(curve, grid)

    if not np.isfinite(dfs).all() or np.any(dfs <= 0):
        raise RuntimeError(f"{name}: nonpositive or nonfinite discount factor.")
    if not np.isclose(dfs[0], 1.0, rtol=0, atol=1e-12):
        raise RuntimeError(f"{name}: D(0) must equal 1.")
    if not np.isfinite(z).all() or not np.isfinite(f).all():
        raise RuntimeError(f"{name}: nonfinite spot or forward rate.")

    axz.plot(grid, 100 * z, color=COLORS[name], label=name)
    if isinstance(curve, TreasuryBootstrapCurve):
        axf.stairs(100 * curve.segment_forwards_, curve.nodes_,
                   color=COLORS[name], label=name, baseline=None)
    else:
        axf.plot(grid, 100 * f, color=COLORS[name], label=name)

    forward_summary[name] = {
        "1Y1Y forward (%)": 100 * period_forward_cc(curve, 1.0, 2.0),
        "Min grid forward (%)": 100 * f.min(),
        "Max grid forward (%)": 100 * f.max(),
    }
    curve_grid[f"{name}_discount_factor"] = dfs
    curve_grid[f"{name}_zero_decimal"] = z
    curve_grid[f"{name}_forward_decimal"] = f

axz.set(title=f"Spot and implied forward curves | {fit_date:%Y-%m-%d}",
        ylabel="Continuous spot rate (%)")
axf.set(xlabel="Maturity (years)", ylabel="Instantaneous forward (%)")
for a in (axz, axf):
    a.grid(alpha=0.25)
    a.legend()
fig.tight_layout()
fig.savefig(curve_dir / "spot_and_forward_curves.png", dpi=180, bbox_inches="tight")
plt.show()

forward_summary = pd.DataFrame.from_dict(forward_summary, orient="index")
forward_summary.to_csv(curve_dir / "forward_summary.csv")
curve_grid.to_csv(curve_dir / "spot_discount_forward_grid.csv", index=False)
display(forward_summary.round(4))

### 2.3 Held-out quote validation

On evenly spaced dates, one interior quote is removed, each model is refitted on the remaining ten, and the removed quote is predicted. Errors are in basis points (model minus observed). Only date–tenor trials where **all three** models succeed enter the comparison, so the models are scored on identical cases. Dates are sampled, not exhaustive, and trials on adjacent dates are not independent.

In [ ]:
panel = core_panel.sort_index()
ordered_ids = sorted(TENORS, key=TENORS.get)
if panel.empty or N_VALIDATION_DATES < 1:
    raise ValueError("Need a nonempty panel and at least one validation date.")
if not HOLDOUT_IDS or not set(HOLDOUT_IDS).issubset(ordered_ids[1:-1]):
    raise ValueError("Choose one or more interior tenors for holdout tests.")
if len(HOLDOUT_IDS) != len(set(HOLDOUT_IDS)):
    raise ValueError("Holdout tenor IDs must be unique.")

n = min(N_VALIDATION_DATES, len(panel))
positions = np.linspace(0, len(panel) - 1, n, dtype=int)
validation_dates = panel.index[positions]
records = []

for k, vdate in enumerate(validation_dates, start=1):
    vrow = panel.loc[vdate]
    for held_id in HOLDOUT_IDS:
        train_tenors = {s: T for s, T in TENORS.items() if s != held_id}
        train_row = vrow.drop(index=held_id)
        observed = float(vrow[held_id])

        for name in MODELS:
            record = {"date": vdate, "held_out": held_id, "model": name,
                      "observed": observed, "predicted": np.nan,
                      "residual_bp": np.nan, "near_bounds": "",
                      "status": "ok", "error": ""}
            try:
                c = fit_curve(name, train_row, train_tenors)
                predicted = quote_from_discount(c.discount_factors, TENORS[held_id])
                if not np.isfinite(predicted):
                    raise RuntimeError("Nonfinite held-out quote.")
                record.update(
                    predicted=predicted,
                    residual_bp=10_000 * (predicted - observed),
                    near_bounds=", ".join(getattr(c, "near_bounds_", [])),
                )
            except (RuntimeError, ValueError, FloatingPointError) as exc:
                record.update(status="failed", error=str(exc))
            records.append(record)

    if k % 6 == 0 or k == len(validation_dates):
        print(f"Validated {k}/{len(validation_dates)} dates.")

validation = pd.DataFrame(records)
validation.to_csv(curve_dir / "held_out_trials.csv", index=False)

status = pd.crosstab(validation["model"], validation["status"])
status = status.reindex(index=MODELS, columns=["ok", "failed"], fill_value=0)
display(status)

successful = validation.loc[validation["status"].eq("ok")]
matched = (
    successful.pivot(index=["date", "held_out"], columns="model", values="residual_bp")
    .reindex(columns=MODELS).dropna()
)
if matched.empty:
    raise RuntimeError("No matched trials. Inspect held_out_trials.csv.")
print(f"Matched trials: {len(matched)} / {len(validation_dates) * len(HOLDOUT_IDS)}")

overall = pd.DataFrame({
    "n": matched.count(),
    "bias_bp": matched.mean(),
    "mae_bp": matched.abs().mean(),
    "rmse_bp": np.sqrt(matched.pow(2).mean()),
    "p95_abs_bp": matched.abs().quantile(0.95),
})
by_tenor_rmse = np.sqrt(matched.pow(2).groupby(level="held_out").mean()).reindex(HOLDOUT_IDS)
by_tenor_rmse.insert(0, "n_matched", matched.groupby(level="held_out").size())
overall.to_csv(curve_dir / "held_out_overall.csv")
by_tenor_rmse.to_csv(curve_dir / "held_out_by_tenor.csv")
display(overall.round(3))
display(by_tenor_rmse.round(3))

fig, axes = plt.subplots(len(HOLDOUT_IDS), 1, figsize=(11, 2.7 * len(HOLDOUT_IDS)),
                         sharex=True, squeeze=False)
for ax, held_id in zip(axes[:, 0], HOLDOUT_IDS):
    if held_id in matched.index.get_level_values("held_out"):
        e = matched.xs(held_id, level="held_out").sort_index()
        for name in MODELS:
            ax.plot(e.index, e[name], marker=".", linewidth=1, color=COLORS[name], label=name)
        ax.legend(loc="best")
    else:
        ax.text(0.5, 0.5, "No matched trials", transform=ax.transAxes, ha="center")
    ax.axhline(0, color="black", linewidth=0.8)
    ax.set(title=f"Held-out {TENORS[held_id]:g}Y quote", ylabel="Error (bp)")
    ax.grid(alpha=0.25)
axes[-1, 0].set_xlabel("Observation date")
fig.tight_layout()
fig.savefig(curve_dir / "held_out_errors.png", dpi=180, bbox_inches="tight")
plt.show()
print(f"Section 2 outputs saved under: {curve_dir}")

## 3. Bond pricing and interest-rate risk

The synthetic portfolio holds three $1,000,000 semiannual fixed-coupon bonds (2Y at 3.5%, 7Y at 4.0%, 20Y at 4.5%), priced by discounting scheduled cash flows on each curve. Two identities guard the pricer: a zero-coupon bond is worth face × D(T), and a bond paying the curve's own par coupon is worth 100.

In [ ]:
@dataclass(frozen=True)
class FixedRateBond:
    name: str
    maturity_years: float
    coupon_rate: float  # annual coupon as a decimal: 4% = 0.04
    face_value: float

    def cash_flows(self):
        values = [self.maturity_years, self.coupon_rate, self.face_value]
        if not np.isfinite(values).all():
            raise ValueError(f"{self.name}: bond inputs must be finite.")
        if self.maturity_years <= 0 or self.face_value <= 0:
            raise ValueError(f"{self.name}: maturity and face value must be positive.")
        if self.coupon_rate < 0:
            raise ValueError(f"{self.name}: coupon cannot be negative.")

        n = int(round(2 * self.maturity_years))
        if n < 1 or not np.isclose(n / 2, self.maturity_years, rtol=0, atol=1e-12):
            raise ValueError(f"{self.name}: maturity must be on the semiannual grid.")

        times = np.arange(1, n + 1) / 2
        amounts = np.full(n, self.face_value * self.coupon_rate / 2)
        amounts[-1] += self.face_value
        return times, amounts


def bond_pv(bond, discount_factors):
    """Dollar PV for a base or shocked discount-factor function."""
    times, amounts = bond.cash_flows()
    dfs = np.asarray(discount_factors(times), dtype=float)
    if dfs.shape != times.shape:
        raise ValueError(f"{bond.name}: expected one discount factor per cash flow.")
    if not np.isfinite(dfs).all() or np.any(dfs <= 0):
        raise ValueError(f"{bond.name}: discount factors must be positive and finite.")
    return float(amounts @ dfs)


portfolio = (
    FixedRateBond("2Y fixed", 2.0, 0.035, 1_000_000),
    FixedRateBond("7Y fixed", 7.0, 0.040, 1_000_000),
    FixedRateBond("20Y fixed", 20.0, 0.045, 1_000_000),
)

par_checks, valuation_rows = [], []
for model, curve in curves.items():
    discount = curve.discount_factors
    for bond in portfolio:
        T = bond.maturity_years

        zero = FixedRateBond(f"{T:g}Y zero check", T, 0.0, 100.0)
        np.testing.assert_allclose(
            bond_pv(zero, discount), 100.0 * discount(T), rtol=0, atol=1e-8,
            err_msg=f"{model}: {T:g}Y zero-coupon check failed.",
        )

        par_coupon = quote_from_discount(discount, T)
        par_price = bond_pv(FixedRateBond(f"{T:g}Y par check", T, par_coupon, 100.0), discount)
        np.testing.assert_allclose(
            par_price, 100.0, rtol=0, atol=1e-8,
            err_msg=f"{model}: {T:g}Y par-pricing check failed.",
        )
        par_checks.append({"model": model, "maturity_years": T,
                           "par_coupon_pct": 100 * par_coupon,
                           "par_price_per_100": par_price})

        pv = bond_pv(bond, discount)
        valuation_rows.append({
            "curve_date": fit_date.strftime("%Y-%m-%d"), "model": model,
            "bond": bond.name, "maturity_years": T,
            "coupon_pct": 100 * bond.coupon_rate, "face_usd": bond.face_value,
            "price_per_100": 100 * pv / bond.face_value, "pv_usd": pv,
        })

bond_values = pd.DataFrame(valuation_rows)
check_table = pd.DataFrame(par_checks)
bond_pv_table = bond_values.pivot(index="bond", columns="model", values="pv_usd")
bond_pv_table = bond_pv_table.reindex(index=[b.name for b in portfolio], columns=list(curves))
bond_pv_table.loc["Portfolio total"] = bond_pv_table.sum(axis=0)

bond_dir = curve_dir.parent / "section_3_bonds"
bond_dir.mkdir(parents=True, exist_ok=True)
bond_values.to_csv(bond_dir / "base_bond_valuations.csv", index=False)
check_table.to_csv(bond_dir / "par_pricing_checks.csv", index=False)

print(f"Valuation date: {fit_date:%Y-%m-%d}")
print(f"Passed {len(check_table)} zero-coupon checks and {len(check_table)} par checks.")
display(bond_pv_table.round(2))

### 3.1 Duration, convexity and key-rate DV01

Shocks are applied to **zero rates**: \(D_{\text{shocked}}(t) = D(t)\,e^{-t\,s(t)}\), where \(s(t)\) interpolates linearly between key-rate anchors and is flat outside them. Parallel risk uses central differences. Key-rate DV01 (KR01) bumps one anchor at a time, so the nine KR01s should sum to parallel DV01. Positive DV01 means a loss when rates rise. Dollar duration and convexity are cross-checked against closed-form discounted cash-flow moments.

In [ ]:
BP = 1e-4
BUMP_BP = 1.0
KEY_YEARS = np.array([0.5, 1.0, 2.0, 3.0, 5.0, 7.0, 10.0, 20.0, 30.0])
KEY_LABELS = ["6M", "1Y", "2Y", "3Y", "5Y", "7Y", "10Y", "20Y", "30Y"]
PORTFOLIO_LABEL = "Portfolio total"


def shocked_discount_factors(base_discount, anchors_years, shocks_bp):
    """Return D_shocked(t) = D_base(t) * exp(-t * shock(t)), shock in decimals."""
    anchors = np.asarray(anchors_years, dtype=float).copy()
    shocks = np.asarray(shocks_bp, dtype=float).copy()
    if anchors.ndim != 1 or shocks.shape != anchors.shape or anchors.size == 0:
        raise ValueError("Supply one shock per anchor on a nonempty 1D grid.")
    if (not np.isfinite(anchors).all() or not np.isfinite(shocks).all()
            or anchors[0] < 0 or np.any(np.diff(anchors) <= 0)):
        raise ValueError("Anchors must be finite, nonnegative, and increasing.")

    def discount(t):
        t = np.asarray(t, dtype=float)
        if np.any(~np.isfinite(t) | (t < 0)):
            raise ValueError("Cash-flow times must be finite and nonnegative.")
        shift = BP * np.interp(t, anchors, shocks)
        return base_discount(t) * np.exp(-t * shift)

    return discount


def portfolio_pvs(discount):
    """PV vector in portfolio order."""
    return np.array([bond_pv(bond, discount) for bond in portfolio])


bond_names = [bond.name for bond in portfolio]
h = BUMP_BP * BP
parallel = np.full(KEY_YEARS.size, BUMP_BP)
key_bumps = BUMP_BP * np.eye(KEY_YEARS.size)
parallel_rows, key_rows = [], []

for model, curve in curves.items():
    base = curve.discount_factors
    p0 = portfolio_pvs(base)

    def bumped_pvs(shocks, base=base):
        return portfolio_pvs(shocked_discount_factors(base, KEY_YEARS, shocks))

    p_up, p_dn = bumped_pvs(parallel), bumped_pvs(-parallel)
    dollar_duration = (p_dn - p_up) / (2 * h)
    dollar_convexity = (p_up + p_dn - 2 * p0) / h**2
    dv01 = dollar_duration * BP
    kr01 = np.column_stack([
        (bumped_pvs(-bump) - bumped_pvs(bump)) / (2 * BUMP_BP) for bump in key_bumps
    ])

    moments = []
    for bond in portfolio:
        t, cf = bond.cash_flows()
        discounted_cf = cf * base(t)
        moments.append((t @ discounted_cf, (t**2) @ discounted_cf))
    exact_dd, exact_dc = np.asarray(moments).T

    np.testing.assert_allclose(bumped_pvs(np.zeros_like(parallel)), p0, rtol=0, atol=1e-8,
                               err_msg=f"{model}: zero shock changed PV.")
    np.testing.assert_allclose(dollar_duration, exact_dd, rtol=2e-6, atol=1e-6,
                               err_msg=f"{model}: duration check failed.")
    np.testing.assert_allclose(dollar_convexity, exact_dc, rtol=2e-6, atol=1e-6,
                               err_msg=f"{model}: convexity check failed.")
    np.testing.assert_allclose(kr01.sum(axis=1), dv01, rtol=5e-6, atol=1e-5,
                               err_msg=f"{model}: key-rate DV01 does not reconcile to parallel DV01.")

    pv_all = np.r_[p0, p0.sum()]
    dd_all = np.r_[dollar_duration, dollar_duration.sum()]
    dc_all = np.r_[dollar_convexity, dollar_convexity.sum()]
    kr_all = np.vstack([kr01, kr01.sum(axis=0)])

    for i, name in enumerate(bond_names + [PORTFOLIO_LABEL]):
        parallel_dv01 = BP * dd_all[i]
        parallel_rows.append({
            "date": fit_date.strftime("%Y-%m-%d"), "model": model, "bond": name,
            "base_pv_usd": pv_all[i],
            "duration_years": dd_all[i] / pv_all[i],
            "convexity_years_sq": dc_all[i] / pv_all[i],
            "parallel_dv01_usd_per_bp": parallel_dv01,
            "key_rate_sum_usd_per_bp": kr_all[i].sum(),
            "key_rate_minus_parallel_usd_per_bp": kr_all[i].sum() - parallel_dv01,
        })
        for tenor, years, value in zip(KEY_LABELS, KEY_YEARS, kr_all[i]):
            key_rows.append({
                "date": fit_date.strftime("%Y-%m-%d"), "model": model, "bond": name,
                "key_tenor": tenor, "key_years": years, "kr01_usd_per_bp": value,
            })

risk_summary = pd.DataFrame(parallel_rows)
key_rate_risk = pd.DataFrame(key_rows)
risk_summary.to_csv(bond_dir / "bond_parallel_risk.csv", index=False)
key_rate_risk.to_csv(bond_dir / "bond_key_rate_risk.csv", index=False)

print("Passed zero-shock, analytical duration/convexity, and DV01 reconciliation checks.")
display(risk_summary.drop(columns="date").round(6))

portfolio_kr01 = (
    key_rate_risk.loc[key_rate_risk["bond"].eq(PORTFOLIO_LABEL)]
    .pivot(index="key_tenor", columns="model", values="kr01_usd_per_bp")
    .reindex(index=KEY_LABELS, columns=list(curves))
)
display(portfolio_kr01.round(2))

ax = portfolio_kr01.plot.bar(figsize=(10, 4.5), width=0.8, rot=0)
ax.set(title=f"Portfolio key-rate DV01 | {fit_date:%Y-%m-%d}",
       xlabel="Zero-rate shock anchor",
       ylabel="USD per bp (positive = loss for an upward bump)")
ax.grid(axis="y", alpha=0.25)
fig = ax.get_figure()
fig.tight_layout()
fig.savefig(bond_dir / "portfolio_key_rate_dv01.png", dpi=180, bbox_inches="tight")
plt.show()

### 3.2 Treasury auction price check

This is an independent, **single-observation** sanity check, not a validation sample. The cell selects the most recent regular 7-year note auction in the search window whose auction date has a complete FRED curve, then compares each model price (at issue settlement) with the published auction price per $100. Time uses actual days / 365 with scheduled coupon dates and no business-day adjustment. FRED and the auction are matched by calendar date, not timestamp, so a gap of a few cents is not by itself evidence of a model error.

In [ ]:
fields = [
    "cusip", "security_type", "security_term", "auction_date", "issue_date",
    "dated_date", "maturity_date", "first_int_payment_date", "int_rate", "price_per100",
]
params = {
    "fields": ",".join(fields),
    "filter": ",".join([
        "security_type:eq:Note", "security_term:eq:7-Year",
        f"auction_date:gte:{AUCTION_START}", f"auction_date:lte:{AUCTION_END}",
    ]),
    "sort": "-auction_date,cusip",
    "page[size]": 100,
}

response = requests.get(AUCTIONS_API, params=params, timeout=(10, 45))
response.raise_for_status()
retrieved_at = datetime.now(timezone.utc).isoformat()
payload = response.json()

rows = payload.get("data", [])
if not rows:
    raise RuntimeError("No 7Y Note auctions returned. Check the search window.")
if payload.get("links", {}).get("next"):
    raise RuntimeError("Auction query has more pages. Narrow the search window.")

auctions = pd.DataFrame(rows)
for col in ["auction_date", "issue_date", "dated_date", "maturity_date", "first_int_payment_date"]:
    auctions[col] = pd.to_datetime(auctions[col], errors="coerce")
for col in ["int_rate", "price_per100"]:
    auctions[col] = pd.to_numeric(auctions[col], errors="coerce")


def regular_coupon_schedule(first, maturity):
    """Semiannual dates from the first coupon to maturity, or None if irregular."""
    months = (maturity.year - first.year) * 12 + maturity.month - first.month
    if months < 0 or months % 6:
        return None
    dates = [maturity - pd.DateOffset(months=6 * i) for i in range(months // 6, -1, -1)]
    if maturity.is_month_end:
        dates = [d + pd.offsets.MonthEnd(0) for d in dates]
    if dates[0] != first:
        return None
    return pd.DatetimeIndex(dates)


candidates = auctions.loc[
    auctions["auction_date"].isin(core_panel.index)
    & auctions["issue_date"].eq(auctions["dated_date"])
    & auctions["auction_date"].le(auctions["issue_date"])
    & auctions["first_int_payment_date"].gt(auctions["issue_date"])
    & auctions["maturity_date"].ge(auctions["first_int_payment_date"])
    & auctions["int_rate"].gt(0)
    & auctions["price_per100"].gt(0)
].sort_values(["auction_date", "cusip"], ascending=[False, True])

selected = None
for _, candidate in candidates.iterrows():
    schedule = regular_coupon_schedule(candidate["first_int_payment_date"], candidate["maturity_date"])
    if schedule is None:
        continue
    maturity = candidate["maturity_date"]
    prev_coupon = maturity - pd.DateOffset(months=6 * len(schedule))  # must equal dated date: no stub
    if maturity.is_month_end:
        prev_coupon += pd.offsets.MonthEnd(0)
    if prev_coupon != candidate["dated_date"]:
        continue
    selected = candidate
    break

if selected is None:
    raise RuntimeError("No qualifying regular 7Y note overlaps the complete FRED dates. "
                       "Adjust the search window; do not substitute a different curve date.")

auction_date = selected["auction_date"]
issue_date = selected["issue_date"]
observed_price = float(selected["price_per100"])
coupon_pct = float(selected["int_rate"])

t_issue = (issue_date - auction_date).days / 365.0
t_pay = (schedule - auction_date).days.to_numpy(dtype=float) / 365.0
if not (0 <= t_issue < t_pay[0] <= t_pay[-1] <= max(TENORS.values())):
    raise RuntimeError("Settlement or payment times fall outside the supported range.")

cashflows = np.full(len(schedule), coupon_pct / 2.0)  # USD per $100 face
cashflows[-1] += 100.0

auction_row = core_panel.loc[auction_date]
auction_comparison = []
for name in MODELS:
    curve = fit_curve(name, auction_row, TENORS)
    d_issue = float(curve.discount_factors(t_issue))
    d_pay = np.asarray(curve.discount_factors(t_pay), dtype=float)
    if (not np.isfinite(d_issue) or d_issue <= 0
            or not np.isfinite(d_pay).all() or np.any(d_pay <= 0)):
        raise RuntimeError(f"{name}: invalid discount factors.")
    model_price = float(cashflows @ (d_pay / d_issue))
    auction_comparison.append({
        "cusip": selected["cusip"],
        "auction_date": auction_date.date().isoformat(),
        "issue_settlement": issue_date.date().isoformat(),
        "maturity_date": selected["maturity_date"].date().isoformat(),
        "coupon_pct": coupon_pct, "model": name,
        "observed_auction_price_per_100": observed_price,
        "model_price_per_100": model_price,
        "model_minus_auction_per_100": model_price - observed_price,
    })

real_price_check = pd.DataFrame(auction_comparison)
real_cashflows = pd.DataFrame({
    "scheduled_payment_date": schedule,
    "years_from_auction": t_pay,
    "cashflow_per_100": cashflows,
})

market_check_dir = bond_dir / "real_treasury_auction_check"
market_check_dir.mkdir(parents=True, exist_ok=True)
(market_check_dir / "raw_auction_response.json").write_bytes(response.content)
real_price_check.to_csv(market_check_dir / "price_comparison.csv", index=False)
real_cashflows.to_csv(market_check_dir / "cashflows.csv", index=False)
(market_check_dir / "metadata.json").write_text(json.dumps({
    "source": "US Treasury Fiscal Data - Treasury Securities Auctions",
    "request_url": response.url,
    "retrieved_at_utc": retrieved_at,
    "price_basis": "Published auction price per $100, at issue settlement",
    "curve_date": auction_date.date().isoformat(),
    "curve_time_basis": "ACT/365 fixed",
    "coupon_schedule": "Regular semiannual scheduled dates; no business-day adjustment",
    "limitation": "Same calendar date, not necessarily the same intraday timestamp",
}, indent=2), encoding="utf-8")

print(f"7Y Treasury {selected['cusip']} | Auction: {auction_date:%Y-%m-%d} | "
      f"Issue settlement: {issue_date:%Y-%m-%d} | Coupon: {coupon_pct:.3f}%")
display(real_price_check[["model", "observed_auction_price_per_100",
                          "model_price_per_100", "model_minus_auction_per_100"]].round(6))

## 4. Stress testing

### 4.1 Engine

Scenarios shock the fitted **zero curve** at the nine key-rate anchors; the cash flows are unchanged. For every scenario the engine reports:

- **Full repricing P&L**, the reference answer.
- **Key-rate estimate**, \(-\sum_k \text{KR01}_k \cdot s_k\), a first-order approximation.
- **Duration + convexity estimate**, parallel shifts only.
- **Error** defined as *estimate − full repricing*, in dollars.

Steepeners and flatteners are specified by their 2Y and 10Y shocks, interpolated between those anchors and held flat outside them. The cell first checks that a zero shock gives zero P&L and that small parallel and twist shocks agree with the approximations.

In [ ]:
stress_dir = bond_dir.parent / "section_4_stress"
stress_dir.mkdir(parents=True, exist_ok=True)
STRESS_POSITIONS = [b.name for b in portfolio] + [PORTFOLIO_LABEL]


def twist_shock(short_bp, long_bp):
    """2Y and 10Y shocks mapped onto the key-rate grid."""
    return np.interp(KEY_YEARS, [2.0, 10.0], [short_bp, long_bp])


def run_stress(model, shock_bp, scenario="Custom"):
    """Reprice the unchanged portfolio and compare P&L approximations."""
    if model not in curves:
        raise ValueError(f"Unknown curve model: {model!r}")
    shock_bp = np.asarray(shock_bp, dtype=float)
    if shock_bp.shape != KEY_YEARS.shape or not np.isfinite(shock_bp).all():
        raise ValueError("Supply one finite shock in bp per key-rate anchor.")

    base_discount = curves[model].discount_factors
    shocked_discount = shocked_discount_factors(base_discount, KEY_YEARS, shock_bp)

    p0 = portfolio_pvs(base_discount)
    p1 = portfolio_pvs(shocked_discount)
    base = np.r_[p0, p0.sum()]
    shocked = np.r_[p1, p1.sum()]
    full_pnl = np.r_[p1 - p0, (p1 - p0).sum()]

    risk = (risk_summary.loc[risk_summary["model"].eq(model)]
            .set_index("bond").reindex(STRESS_POSITIONS))
    kr = (key_rate_risk.loc[key_rate_risk["model"].eq(model)]
          .pivot(index="bond", columns="key_years", values="kr01_usd_per_bp")
          .reindex(index=STRESS_POSITIONS, columns=KEY_YEARS))

    np.testing.assert_allclose(base, risk["base_pv_usd"].to_numpy(), rtol=0, atol=1e-6,
                               err_msg="Base PV differs from Section 3.1. Rerun sensitivities first.")
    if not np.isfinite(kr.to_numpy()).all():
        raise RuntimeError("Missing key-rate DV01. Rerun Section 3.1.")

    key_rate_pnl = -(kr.to_numpy() @ shock_bp)
    duration_convexity_pnl = np.full(len(base), np.nan)
    if np.all(shock_bp == shock_bp[0]):
        dy = shock_bp[0] * BP
        duration_convexity_pnl = base * (
            -risk["duration_years"].to_numpy() * dy
            + 0.5 * risk["convexity_years_sq"].to_numpy() * dy**2
        )

    return pd.DataFrame({
        "date": fit_date.strftime("%Y-%m-%d"), "scenario": scenario, "model": model,
        "bond": STRESS_POSITIONS, "base_pv_usd": base, "shocked_pv_usd": shocked,
        "full_pnl_usd": full_pnl, "full_pnl_pct": 100 * full_pnl / base,
        "key_rate_pnl_usd": key_rate_pnl,
        "key_rate_error_usd": key_rate_pnl - full_pnl,
        "duration_convexity_pnl_usd": duration_convexity_pnl,
        "duration_convexity_error_usd": duration_convexity_pnl - full_pnl,
    })


def plot_stress(model, shock_bp, scenario):
    """Base and shocked zero curves, plus the shock applied between anchors."""
    t = np.unique(np.r_[np.linspace(min(TENORS.values()), max(TENORS.values()), 700), KEY_YEARS])
    base_pct = 100 * curves[model].zero_rates(t)
    shift_bp = np.interp(t, KEY_YEARS, shock_bp)

    fig, (ax, sx) = plt.subplots(2, 1, figsize=(10, 6), sharex=True,
                                 gridspec_kw={"height_ratios": [2, 1]})
    ax.plot(t, base_pct, label="Base", linewidth=2)
    ax.plot(t, base_pct + shift_bp / 100, label="Shocked", linewidth=2)
    ax.set(title=f"{model} | {scenario} | {fit_date:%Y-%m-%d}", ylabel="Zero rate (%)")
    ax.legend()
    sx.plot(t, shift_bp, color="tab:purple")
    sx.scatter(KEY_YEARS, shock_bp, color="tab:purple", s=22, zorder=3)
    sx.axhline(0, color="black", linewidth=0.7)
    sx.set(xlabel="Maturity (years)", ylabel="Shock (bp)")
    for a in (ax, sx):
        a.grid(alpha=0.25)
    fig.tight_layout()
    return fig


# ---- Engine checks ---------------------------------------------------------
for model in curves:
    zero = run_stress(model, np.zeros_like(KEY_YEARS), "Zero check")
    np.testing.assert_allclose(
        zero[["full_pnl_usd", "key_rate_pnl_usd", "duration_convexity_pnl_usd"]].to_numpy(),
        0.0, rtol=0, atol=1e-8, err_msg=f"{model}: zero shock did not produce zero P&L.")

    small_parallel = run_stress(model, np.full_like(KEY_YEARS, 0.1), "Small parallel check")
    np.testing.assert_allclose(
        small_parallel["duration_convexity_pnl_usd"], small_parallel["full_pnl_usd"],
        rtol=0, atol=0.01, err_msg=f"{model}: small parallel duration/convexity check failed.")

    small_twist = run_stress(model, twist_shock(0.1, -0.1), "Small twist check")
    np.testing.assert_allclose(
        small_twist["key_rate_pnl_usd"], small_twist["full_pnl_usd"],
        rtol=0, atol=0.10, err_msg=f"{model}: small twist key-rate check failed.")

print("Passed zero-shock, small parallel, and small twist checks for all curves.")

# ---- Default scenarios (illustrative) -----------------------------------------
custom_20y = np.zeros_like(KEY_YEARS)
custom_20y[KEY_YEARS == 20.0] = 75.0

SCENARIOS = {
    "Parallel +100 bp": np.full_like(KEY_YEARS, 100.0),
    "Parallel -100 bp": np.full_like(KEY_YEARS, -100.0),
    "Steepener (2Y -25 / 10Y +50)": twist_shock(-25.0, 50.0),
    "Flattener (2Y +50 / 10Y -25)": twist_shock(50.0, -25.0),
    "Custom 20Y +75 bp": custom_20y,
}

stress_results = pd.concat(
    [run_stress(m, s, name) for name, s in SCENARIOS.items() for m in curves],
    ignore_index=True,
)
stress_results.to_csv(stress_dir / "stress_by_bond_and_portfolio.csv", index=False)

scenario_grid = pd.DataFrame.from_dict(SCENARIOS, orient="index", columns=KEY_LABELS)
scenario_grid.index.name = "scenario"
scenario_grid.to_csv(stress_dir / "scenario_shocks_bp.csv")

portfolio_stress = stress_results.loc[stress_results["bond"].eq(PORTFOLIO_LABEL)].reset_index(drop=True)
portfolio_stress.to_csv(stress_dir / "portfolio_stress_summary.csv", index=False)

display(portfolio_stress[[
    "scenario", "model", "full_pnl_usd", "full_pnl_pct", "key_rate_pnl_usd",
    "key_rate_error_usd", "duration_convexity_pnl_usd", "duration_convexity_error_usd",
]].round(2))

example = "Flattener (2Y +50 / 10Y -25)"
fig = plot_stress("NS", SCENARIOS[example], example)
fig.savefig(stress_dir / "flattener_zero_curve.png", dpi=180, bbox_inches="tight")
plt.show()

pnl_table = portfolio_stress.pivot(index="scenario", columns="model", values="full_pnl_usd")
pnl_table = pnl_table.reindex(index=list(SCENARIOS), columns=list(curves))
ax = pnl_table.plot.barh(figsize=(10, 5))
ax.axvline(0, color="black", linewidth=0.8)
ax.xaxis.set_major_formatter(StrMethodFormatter("${x:,.0f}"))
ax.set(title=f"Portfolio stress P&L by curve model | {fit_date:%Y-%m-%d}",
       xlabel="Full repricing P&L (USD)", ylabel="")
ax.grid(axis="x", alpha=0.25)
fig = ax.get_figure()
fig.tight_layout()
fig.savefig(stress_dir / "portfolio_stress_pnl.png", dpi=180, bbox_inches="tight")
plt.show()
print(f"Saved stress outputs to: {stress_dir}")

### 4.2 Interactive playground

Choose a curve model and scenario, adjust the sliders, and click **Run stress**. The widget calls `run_stress` from 4.1, so results match the saved tables for identical inputs. *Custom* exposes all nine anchors. The valuation date is `FIT_DATE`. The playground is optional for the export in Section 5. It needs a live Colab or Jupyter session and will not render on GitHub.

In [ ]:
import ipywidgets as widgets

try:
    from google.colab import output as colab_output
except ImportError:
    pass
else:
    colab_output.enable_custom_widget_manager()

model_pick = widgets.Dropdown(options=list(curves), value="NS", description="Curve:")
scenario_pick = widgets.Dropdown(
    options=["Parallel", "Steepener", "Flattener", "Custom"],
    value="Parallel", description="Scenario:",
)


def bp_slider(label, value=0):
    return widgets.IntSlider(
        value=value, min=-300, max=300, step=1, description=label,
        continuous_update=False, layout=widgets.Layout(width="340px"),
    )


parallel_slider = bp_slider("All (bp)", 100)
short_slider = bp_slider("2Y (bp)", -25)
long_slider = bp_slider("10Y (bp)", 50)
custom_sliders = [bp_slider(f"{label} (bp)", 75 if label == "20Y" else 0) for label in KEY_LABELS]

controls = widgets.VBox()
run_button = widgets.Button(description="Run stress", button_style="primary")
out = widgets.Output()


def update_controls(change=None):
    scenario = scenario_pick.value
    if scenario == "Parallel":
        controls.children = (parallel_slider,)
    elif scenario == "Custom":
        controls.children = tuple(custom_sliders)
    else:
        short_slider.value, long_slider.value = (-25, 50) if scenario == "Steepener" else (50, -25)
        controls.children = (short_slider, long_slider)


def render_stress(_=None):
    model, scenario = model_pick.value, scenario_pick.value
    if scenario == "Parallel":
        shock = np.full_like(KEY_YEARS, parallel_slider.value)
    elif scenario == "Custom":
        shock = np.array([s.value for s in custom_sliders], dtype=float)
    else:
        shock = twist_shock(short_slider.value, long_slider.value)

    with out:
        clear_output(wait=True)
        spread_change = np.interp(10.0, KEY_YEARS, shock) - np.interp(2.0, KEY_YEARS, shock)
        print(f"Change in 10Y-2Y zero-rate spread: {spread_change:+.0f} bp")
        if scenario == "Steepener" and spread_change <= 0:
            print("These slider values do not steepen the 2Y-10Y segment.")
        elif scenario == "Flattener" and spread_change >= 0:
            print("These slider values do not flatten the 2Y-10Y segment.")

        result = run_stress(model, shock, scenario)
        cols = ["base_pv_usd", "shocked_pv_usd", "full_pnl_usd", "full_pnl_pct",
                "key_rate_pnl_usd", "key_rate_error_usd"]
        if np.all(shock == shock[0]):
            cols += ["duration_convexity_pnl_usd", "duration_convexity_error_usd"]
        display(result.set_index("bond")[cols].style.format("{:,.2f}"))
        fig = plot_stress(model, shock, scenario)
        plt.show()
        plt.close(fig)


scenario_pick.observe(update_controls, names="value")
run_button.on_click(render_stress)
update_controls()

display(widgets.VBox([widgets.HBox([model_pick, scenario_pick]), controls, run_button, out]))
render_stress()

## 5. Publication figures and export bundle

This cell redraws eight report figures in one consistent style and saves each as a vector PDF (for LaTeX) and a 300-dpi PNG (for GitHub). It then copies the full data snapshot, writes a figure index, an export metadata file and a SHA-256 inventory, builds a ZIP and verifies its contents. Figures 07 and 08 use the **default scenarios**, never the last widget state. In Figure 07 the red and blue shading show where the shocked curve lies above or below the base curve; it is the scenario difference, **not** a confidence band.

The bundle contains outputs, not this notebook. Download the `.ipynb` from Colab separately.

In [ ]:
required = ("run_dir", "core_panel", "TENORS", "curves", "fit_date", "fit_quotes",
            "instantaneous_forward", "matched", "HOLDOUT_IDS", "portfolio_kr01",
            "real_price_check", "SCENARIOS", "portfolio_stress", "KEY_YEARS",
            "curve_dir", "bond_dir", "stress_dir")
missing = [name for name in required if name not in globals()]
if missing:
    raise RuntimeError("Run the earlier notebook cells first. Missing: " + ", ".join(missing))
if REPORT_MODEL not in curves:
    raise ValueError(f"Unknown REPORT_MODEL: {REPORT_MODEL}")

source = Path(run_dir).resolve()
source_dirs = [source / "raw", Path(curve_dir).resolve(), Path(bond_dir).resolve(), Path(stress_dir).resolve()]
source_files = [source / f for f in ("manifest.json", "yields_percent.csv", "yields_decimal.csv", "tenor_coverage.csv")]
missing_paths = [str(p) for p in source_dirs + source_files if not p.exists()]
if missing_paths:
    raise FileNotFoundError("Missing inputs:\n" + "\n".join(missing_paths))

export_root = Path("exports").resolve()
if export_root == source or source in export_root.parents:
    raise ValueError("Set exports outside the source snapshot directory.")
export_root.mkdir(parents=True, exist_ok=True)
stamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
bundle = export_root / f"yield_curve_report_{fit_date:%Y%m%d}_{stamp}"
fig_dir = bundle / "figures"
fig_dir.mkdir(parents=True)

# Preserve relative paths so the FRED manifest stays valid inside the bundle.
shutil.copytree(source, bundle / "run_snapshot")

models = list(curves)
colors = dict(zip(models, ["#0072B2", "#D55E00", "#009E73"]))
patterns = dict(zip(models, ["-", "--", "-."]))
markers = dict(zip(models, ["o", "s", "^"]))
up, down = "#B43C39", "#2166AC"
asof = pd.Timestamp(fit_date).strftime("%Y-%m-%d")

ids = sorted(TENORS, key=TENORS.get)
maturities = np.array([TENORS[s] for s in ids])
labels = [s[3:].replace("MO", "M") if s.endswith("MO") else s[3:] + "Y" for s in ids]
usd = StrMethodFormatter("${x:,.0f}")

style = {
    "font.family": "DejaVu Sans", "font.size": 10,
    "axes.titlesize": 11, "axes.labelsize": 10,
    "axes.spines.top": False, "axes.spines.right": False,
    "axes.grid": True, "axes.axisbelow": True,
    "grid.color": "#D5DBE1", "grid.linewidth": 0.6,
    "legend.frameon": False,
    "figure.facecolor": "white", "savefig.facecolor": "white",
    "pdf.fonttype": 42,
}
figure_index = []


def save_figure(fig, stem, caption):
    for suffix in ("pdf", "png"):
        fig.savefig(fig_dir / f"{stem}.{suffix}", dpi=300, bbox_inches="tight", pad_inches=0.12)
    figure_index.append({"stem": stem, "caption": caption})
    plt.close(fig)


def quote_axis(ax):
    ax.set_xscale("log")
    ax.set_xticks(maturities, labels)
    ax.minorticks_off()
    ax.set_xlabel("Quoted maturity (log scale)")


with mpl.rc_context(style):

    # 01 - FRED history (missing quotes are not imputed)
    fig, ax = plt.subplots(figsize=(9, 4.2), layout="constrained")
    raw_panel = pd.read_csv(source / "yields_percent.csv", index_col="date", parse_dates=True).sort_index()
    for s, c in zip(["DGS3MO", "DGS2", "DGS10", "DGS30"], ["#7A5195", "#0072B2", "#D55E00", "#009E73"]):
        ax.plot(raw_panel.index, raw_panel[s], color=c, lw=1.05, label=labels[ids.index(s)])
    ax.set(title="U.S. Treasury yields over time", xlabel="Observation date", ylabel="Quoted yield (%)")
    ax.legend(ncol=4)
    save_figure(fig, "01_fred_yield_history",
                "Selected FRED constant-maturity series. Missing quotes are not imputed.")

    # 02 - Quote fit and signed residuals
    fig, (ax, er) = plt.subplots(2, 1, figsize=(9, 6.2), layout="constrained",
                                 gridspec_kw={"height_ratios": [2, 1]})
    x = np.arange(len(ids))
    obs = fit_quotes.xs(models[0], level="model").reindex(ids)
    for j, model in enumerate(models):
        d = fit_quotes.xs(model, level="model").reindex(ids)
        ax.plot(maturities, 100 * d["fitted"], color=colors[model], ls=patterns[model], lw=1.8, label=model)
        er.bar(x + (j - 1) * 0.24, d["residual_bp"], width=0.23, color=colors[model])
    ax.scatter(maturities, 100 * obs["observed"], facecolors="white", edgecolors="#20262D",
               linewidths=1.1, s=32, zorder=5, label="FRED observed")
    quote_axis(ax)
    ax.set(title=f"A. Observed and fitted quotes | {asof}", ylabel="Yield (%)")
    ax.legend(ncol=4)
    er.axhline(0, color="#20262D", lw=0.8)
    er.set_xticks(x, labels)
    er.set(title="B. Quote residuals", xlabel="Quoted tenor", ylabel="Fitted - observed (bp)")
    save_figure(fig, "02_quote_fit_and_residuals",
                "Fitted and observed quotes at the calibration tenors. Bars are signed residuals, not confidence intervals.")

    # 03 - Zero curves and forwards
    t = np.linspace(0, maturities[-1], 1401)
    fig, (az, af) = plt.subplots(2, 1, figsize=(9, 6.1), sharex=True, layout="constrained")
    for model, curve in curves.items():
        az.plot(t, 100 * curve.zero_rates(t), color=colors[model], ls=patterns[model], lw=1.8, label=model)
        if isinstance(curve, TreasuryBootstrapCurve):
            af.stairs(100 * curve.segment_forwards_, curve.nodes_, baseline=None, color=colors[model], lw=1.5)
        else:
            af.plot(t, 100 * instantaneous_forward(curve, t), color=colors[model], ls=patterns[model], lw=1.8)
    az.set(title=f"A. Zero curves | {asof}", ylabel="CC zero rate (%)")
    af.set(title="B. Implied instantaneous forwards", xlabel="Maturity (years)", ylabel="Forward rate (%)")
    az.legend(ncol=3)
    save_figure(fig, "03_zero_and_forward_curves",
                "Zero rates are continuously compounded (CC). Bootstrap forwards are steps. Implied forwards are not forecasts.")

    # 04 - Held-out errors
    fig, axes = plt.subplots(len(HOLDOUT_IDS), 1, figsize=(9, 2.15 * len(HOLDOUT_IDS) + 0.7),
                             sharex=True, squeeze=False, layout="constrained")
    for j, (ax, sid) in enumerate(zip(axes[:, 0], HOLDOUT_IDS)):
        if sid in matched.index.get_level_values("held_out"):
            e = matched.xs(sid, level="held_out").sort_index()
            for model in models:
                ax.plot(e.index, e[model], color=colors[model], ls=patterns[model],
                        marker=markers[model], ms=3, lw=1.1, label=model)
            if j == 0:
                ax.legend(ncol=3)
            ax.text(0.99, 0.93, f"n = {len(e)} matched dates", ha="right", va="top",
                    transform=ax.transAxes, bbox=dict(facecolor="white", edgecolor="none", alpha=0.9))
        else:
            ax.text(0.5, 0.5, "No matched trials", ha="center", transform=ax.transAxes)
        ax.axhline(0, color="#20262D", lw=0.8)
        ax.set(title=f"Held-out {TENORS[sid]:g}Y quote", ylabel="Error (bp)")
    axes[-1, 0].set_xlabel("Observation date")
    save_figure(fig, "04_held_out_errors",
                f"Matched held-out errors across {len(matched)} date-tenor trials. Lines connect sampled dates, not daily observations.")

    # 05 - Key-rate DV01
    fig, ax = plt.subplots(figsize=(9, 4.2), layout="constrained")
    kr = portfolio_kr01.reindex(columns=models)
    x = np.arange(len(kr))
    for j, model in enumerate(models):
        ax.bar(x + (j - 1) * 0.23, kr[model], width=0.22, color=colors[model], label=model)
    ax.set_xticks(x, kr.index)
    ax.set(title=f"Portfolio key-rate DV01 | {asof}", xlabel="Zero-rate shock anchor", ylabel="USD per bp")
    ax.yaxis.set_major_formatter(StrMethodFormatter("{x:,.0f}"))
    ax.legend(ncol=3)
    save_figure(fig, "05_portfolio_key_rate_dv01",
                "Positive exposure denotes an approximate dollar loss for a 1 bp upward bump at the selected anchor.")

    # 06 - Treasury auction price check
    price = real_price_check.set_index("model").reindex(models)
    auction_date_label = str(price["auction_date"].iloc[0])
    cusip = str(price["cusip"].iloc[0])
    observed = float(price["observed_auction_price_per_100"].iloc[0])
    gaps = price["model_minus_auction_per_100"]
    fig, ax = plt.subplots(figsize=(8, 3.6), layout="constrained")
    ax.barh(models, gaps, color=[colors[m] for m in models])
    ax.axvline(0, color="#20262D", lw=0.8)
    ax.invert_yaxis()
    ax.margins(x=0.22)
    for bar, value in zip(ax.patches, gaps):
        ax.annotate(f"{value:+.4f}", (value, bar.get_y() + bar.get_height() / 2),
                    xytext=(5 if value >= 0 else -5, 0), textcoords="offset points",
                    ha="left" if value >= 0 else "right", va="center")
    ax.set(title=f"Treasury {cusip} | Auction {auction_date_label}",
           xlabel="Model minus auction (USD per $100 face)")
    save_figure(fig, "06_treasury_auction_price_check",
                f"Published auction price: {observed:.6f} per $100 face, compared at issue settlement. One auction, not a secondary-market validation sample.")

    # 07 - Scenario curve shapes (shared vertical scale)
    cases = [(name, np.asarray(shock, float)) for name, shock in SCENARIOS.items()]
    t = np.unique(np.r_[np.linspace(maturities[0], maturities[-1], 1000), KEY_YEARS])
    base = 100 * curves[REPORT_MODEL].zero_rates(t)
    deltas = [np.interp(t, KEY_YEARS, s) for _, s in cases]
    y_all = np.concatenate([base] + [base + d / 100 for d in deltas])
    pad = max(0.10, 0.07 * np.ptp(y_all))
    y_lim = (y_all.min() - pad, y_all.max() + pad)
    ncols = 2
    nrows = (len(cases) + ncols - 1) // ncols
    fig, axes = plt.subplots(nrows, ncols, figsize=(10.4, 2.8 * nrows + 0.8),
                             sharex=True, sharey=True, squeeze=False, layout="constrained")
    for j, (ax, (name, shock), delta) in enumerate(zip(axes.flat, cases, deltas)):
        stressed = base + delta / 100
        ax.fill_between(t, base, stressed, where=delta >= 0, interpolate=True, color=up, alpha=0.17, linewidth=0)
        ax.fill_between(t, base, stressed, where=delta <= 0, interpolate=True, color=down, alpha=0.17, linewidth=0)
        ax.plot(t, base, color="#20262D", lw=1.7)
        ax.plot(t, stressed, color="#784C94", lw=1.7, ls="--")
        spread = np.interp(10, KEY_YEARS, shock) - np.interp(2, KEY_YEARS, shock)
        row_ = portfolio_stress.loc[portfolio_stress["scenario"].eq(name)
                                    & portfolio_stress["model"].eq(REPORT_MODEL)]
        pnl = float(row_["full_pnl_usd"].iloc[0])
        pnl_label = f"{'-' if pnl < 0 else '+'}${abs(pnl):,.0f}"
        ax.set_title(f"{chr(65 + j)}. {name}", loc="left")
        ax.set_ylim(y_lim)
        ax.tick_params(labelbottom=True)
        ax.set_xlabel("Maturity (years)")
        if j % ncols == 0:
            ax.set_ylabel("Zero rate (%)")
        ax.text(0.98, 0.04, f"10Y-2Y spread: {spread:+.0f} bp\nPortfolio P&L: {pnl_label}",
                transform=ax.transAxes, ha="right", va="bottom", fontsize=8.5,
                bbox=dict(facecolor="white", edgecolor="#D5DBE1", alpha=0.94, pad=4))
    for ax in list(axes.flat)[len(cases):]:
        ax.axis("off")
    axes.flat[0].legend(handles=[
        Line2D([0], [0], color="#20262D", lw=1.7, label="Base"),
        Line2D([0], [0], color="#784C94", lw=1.7, ls="--", label="Shocked"),
        Patch(facecolor=up, alpha=0.17, label="Rates higher"),
        Patch(facecolor=down, alpha=0.17, label="Rates lower"),
    ], ncol=2, fontsize=8, loc="upper right")
    fig.suptitle(f"Zero-curve stress scenarios | {REPORT_MODEL} | {asof}", fontsize=14)
    save_figure(fig, "07_stress_curve_shapes",
                "All panels share a vertical scale. Shading shows where the shocked zero curve is above (red) or below (blue) the base curve; it is not a confidence band. P&L is full repricing.")

    # 08 - P&L and approximation error
    fig, (ap, ae) = plt.subplots(1, 2, figsize=(12.2, 4.8), sharey=True, layout="constrained",
                                 gridspec_kw={"width_ratios": [1.1, 1]})
    order = list(SCENARIOS)
    y = np.arange(len(order))
    for j, model in enumerate(models):
        d = portfolio_stress.loc[portfolio_stress["model"].eq(model)].set_index("scenario").reindex(order)
        yy = y + (j - 1) * 0.22
        ap.barh(yy, d["full_pnl_usd"], height=0.19, color=colors[model], label=model)
        ae.scatter(d["key_rate_error_usd"], yy, color=colors[model], marker=markers[model], s=32, zorder=4)
    ap.set_yticks(y, order)
    ap.invert_yaxis()
    ap.set(title="A. Full-repricing P&L", xlabel="Portfolio P&L (USD)")
    ae.set(title="B. Key-rate approximation error", xlabel="Estimate minus full repricing (USD)")
    ap.legend(ncol=3)
    for ax in (ap, ae):
        ax.axvline(0, color="#20262D", lw=0.8)
        ax.xaxis.set_major_formatter(usd)
    save_figure(fig, "08_stress_pnl_and_approximation_error",
                "Full-repricing P&L and signed key-rate approximation error for the synthetic portfolio. Parallel duration-plus-convexity results are in the stress CSV.")

# ---- Index, metadata, inventory, ZIP ------------------------------------------
pd.DataFrame(figure_index).to_csv(bundle / "figure_index.csv", index=False)
metadata = {
    "exported_at_utc": datetime.now(timezone.utc).isoformat(),
    "source_snapshot": source.name,
    "curve_date": asof,
    "stress_shape_figure_model": REPORT_MODEL,
    "models": models,
    "complete_curve_dates": len(core_panel),
    "matched_holdout_trials": len(matched),
    "auction_cusip": cusip,
    "auction_date": auction_date_label,
    "python": sys.version.split()[0],
    "packages": {p: importlib_metadata.version(p) for p in ("numpy", "pandas", "scipy", "matplotlib", "requests")},
}
(bundle / "export_metadata.json").write_text(json.dumps(metadata, indent=2), encoding="utf-8")
(bundle / "CONTENTS.md").write_text(
    "# Yield curve report outputs\n\n"
    "- `figures/`: publication PDFs and 300-dpi PNGs.\n"
    "- `figure_index.csv`: source captions for README and LaTeX.\n"
    "- `run_snapshot/`: raw FRED CSVs, coverage, results, and auction evidence.\n"
    "- `export_metadata.json`: run and dependency information.\n"
    "- `inventory_sha256.csv`: file sizes and SHA-256 hashes; excludes itself.\n\n"
    "FRED constant-maturity quotes are calibration inputs, not zero rates. "
    "The stress portfolio is synthetic, and its shocks are illustrative. "
    "The separate market-price check uses one published Treasury auction.\n\n"
    "This bundle contains outputs, not the notebook source. "
    "Export the notebook separately from Colab for the GitHub repo.\n",
    encoding="utf-8",
)

inventory = [
    {"path": p.relative_to(bundle).as_posix(), "bytes": p.stat().st_size,
     "sha256": hashlib.sha256(p.read_bytes()).hexdigest()}
    for p in sorted(bundle.rglob("*")) if p.is_file()
]
pd.DataFrame(inventory).to_csv(bundle / "inventory_sha256.csv", index=False)

zip_path = Path(shutil.make_archive(str(bundle), "zip", root_dir=bundle.parent, base_dir=bundle.name))
with zipfile.ZipFile(zip_path) as z:
    expected = {f"{bundle.name}/{p.relative_to(bundle).as_posix()}" for p in bundle.rglob("*") if p.is_file()}
    actual = {i.filename for i in z.infolist() if not i.is_dir()}
    if actual != expected or z.testzip() is not None:
        raise RuntimeError("ZIP verification failed; do not use this archive.")

print(f"Verified ZIP: {zip_path.name}")
print(f"{len(figure_index)} figures x 2 formats | {len(actual)} archived files")
print(f"ZIP size: {zip_path.stat().st_size / 2**20:.2f} MiB")
display(pd.DataFrame(figure_index))

if DOWNLOAD_ZIP:
    try:
        from google.colab import files
    except ImportError:
        print(f"Not in Colab; ZIP saved at {zip_path}")
    else:
        files.download(str(zip_path))

## Limitations and interpretation

- **Quotes, not zero rates.** FRED constant-maturity yields are par-style quotes. The zero curves are model constructions and depend on the quote-to-curve mapping stated above.
- **Interpolation matters.** The bootstrap reprices every quote exactly by construction; a perfect in-sample fit says nothing about out-of-sample behavior, which is why Section 2.3 holds quotes out.
- **Validation is sampled.** Held-out trials use evenly spaced dates and three interior tenors. Adjacent dates are correlated, so trial counts overstate independent evidence.
- **One auction.** The auction comparison is a single sanity check on one calendar date, not a pricing-accuracy study.
- **Synthetic portfolio and illustrative shocks.** Results show how the machinery behaves, not a view on any real book or a regulatory scenario.
- **Key-rate approximation.** KR01 is first-order. Large or sharply non-parallel shocks leave visible error versus full repricing; report that error rather than assuming it is small.
- **Single valuation date.** Sections 3 and 4 use `FIT_DATE` only.

## References

- Nelson, C. R. and Siegel, A. F. (1987). Parsimonious modeling of yield curves. *Journal of Business*, 60(4), 473–489.
- Svensson, L. E. O. (1994). Estimating and interpreting forward interest rates: Sweden 1992–1994. NBER Working Paper 4871.
- Ho, T. S. Y. (1992). Key rate durations: Measures of interest rate risks. *Journal of Fixed Income*, 2(2), 29–44.
- Federal Reserve Board, H.15 Selected Interest Rates, via FRED (Federal Reserve Bank of St. Louis).
- U.S. Department of the Treasury, Fiscal Data: Treasury Securities Auctions Data.